# Multi-File PHASE Bin Folder Demo

This notebook reads multiple PHASE `.bin` files from one folder, concatenates them along the frame axis, and plots the waveform, PSD, and space-time view for a selected point.

In [1]:
from pathlib import Path
import importlib
%matplotlib qt

import matplotlib.pyplot as plt
import numpy as np
import phase_bin_tools as pbt

pbt = importlib.reload(pbt)

compute_point_psd = pbt.compute_point_psd
extract_point_waveform = pbt.extract_point_waveform
infer_points_per_frame_from_filename = pbt.infer_points_per_frame_from_filename
infer_scan_rate_hz_from_filename = pbt.infer_scan_rate_hz_from_filename
plot_point_psd = pbt.plot_point_psd
plot_point_waveform = pbt.plot_point_waveform
plot_space_time = pbt.plot_space_time
read_phase_bin_folder = pbt.read_phase_bin_folder
read_phase_bin_folder_raw = pbt.read_phase_bin_folder_raw

plt.rcParams['figure.dpi'] = 120

In [7]:
# Cell 1: configure folder, matching rule, and display parameters
folder_path = Path(r"F:\eDAS三次敲击定位数据\3")
file_pattern = "000*.bin"
points_per_frame = None
scan_rate_hz = None
point_index = 75

space_time_frame_slice = slice(None)
space_time_point_slice = slice(None)
space_time_vmin = -0.2
space_time_vmax = 0.2

print('folder_path:', folder_path)
print('file_pattern:', file_pattern)
print('point_index:', point_index)

folder_path: F:\eDAS三次敲击定位数据\3
file_pattern: 000*.bin
point_index: 75


In [8]:
# Cell 2: read and concatenate raw int32 data from all matched files
frame_data_raw, matched_files = read_phase_bin_folder_raw(
    folder_path,
    pattern=file_pattern,
    points_per_frame=points_per_frame,
)

if points_per_frame is None:
    points_per_frame = infer_points_per_frame_from_filename(matched_files[0])
if scan_rate_hz is None:
    scan_rate_hz = infer_scan_rate_hz_from_filename(matched_files[0])

print('matched file count:', len(matched_files))
for path in matched_files:
    print('  ', path.name)
print('points_per_frame:', points_per_frame)
print('scan_rate_hz:', scan_rate_hz)
print('raw shape:', frame_data_raw.shape)
print('raw dtype:', frame_data_raw.dtype)
print('raw min/max:', frame_data_raw.min(), frame_data_raw.max())

matched file count: 283
   0001603-eDAS-2000Hz-0130pt-20260322T155518.691.bin
   0001604-eDAS-2000Hz-0130pt-20260322T155519.690.bin
   0001605-eDAS-2000Hz-0130pt-20260322T155520.691.bin
   0001606-eDAS-2000Hz-0130pt-20260322T155521.692.bin
   0001607-eDAS-2000Hz-0130pt-20260322T155522.683.bin
   0001608-eDAS-2000Hz-0130pt-20260322T155523.682.bin
   0001609-eDAS-2000Hz-0130pt-20260322T155524.684.bin
   0001610-eDAS-2000Hz-0130pt-20260322T155525.694.bin
   0001611-eDAS-2000Hz-0130pt-20260322T155526.692.bin
   0001612-eDAS-2000Hz-0130pt-20260322T155527.690.bin
   0001613-eDAS-2000Hz-0130pt-20260322T155528.683.bin
   0001614-eDAS-2000Hz-0130pt-20260322T155529.683.bin
   0001615-eDAS-2000Hz-0130pt-20260322T155530.676.bin
   0001616-eDAS-2000Hz-0130pt-20260322T155531.677.bin
   0001617-eDAS-2000Hz-0130pt-20260322T155532.686.bin
   0001618-eDAS-2000Hz-0130pt-20260322T155533.688.bin
   0001619-eDAS-2000Hz-0130pt-20260322T155534.680.bin
   0001620-eDAS-2000Hz-0130pt-20260322T155535.691.bin
   0

In [9]:
# Cell 3: read the same folder directly as phase in radians
frame_data_phase, matched_files_phase = read_phase_bin_folder(
    folder_path,
    pattern=file_pattern,
    points_per_frame=points_per_frame,
)

print('same matched files:', matched_files_phase == matched_files)
print('phase shape:', frame_data_phase.shape)
print('phase dtype:', frame_data_phase.dtype)
print('phase min/max (rad):', frame_data_phase.min(), frame_data_phase.max())

same matched files: True
phase shape: (566000, 130)
phase dtype: float64
phase min/max (rad): -53.74206838876454 53.5504108151531


In [10]:
# Cell 4: extract one spatial point waveform from the merged data
waveform = extract_point_waveform(frame_data_phase, point_index)
print('waveform length:', waveform.shape[0])
waveform[:10]

# Cell 5: compute PSD of the selected point
freq_hz, psd_db = compute_point_psd(waveform, sample_rate_hz=scan_rate_hz)
print('freq bins:', freq_hz.shape[0])
print('psd range (dB):', psd_db.min(), psd_db.max())


# Cell 6: plot time-domain waveform and PSD
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_point_waveform(
    waveform,
    sample_rate_hz=scan_rate_hz,
    title=f'Merged Point Waveform (point={point_index})',
    ylabel='Phase (rad)',
    ax=axes[0],
)
plot_point_psd(
    freq_hz,
    psd_db,
    title=f'Merged Point PSD (point={point_index})',
    ax=axes[1],
)
plt.tight_layout()

waveform length: 566000
freq bins: 283001
psd range (dB): -106.87959007786986 12.315474126201543


In [55]:
fig, axes = plt.subplots(4, 1, figsize=(12, 8))


plot_point_waveform(  extract_point_waveform(frame_data_phase, 58),    sample_rate_hz=scan_rate_hz, title=f'Merged Point Waveform (point={73})',  ylabel='Phase (rad)',  ax=axes[0],)
plot_point_waveform(  extract_point_waveform(frame_data_phase, 59),    sample_rate_hz=scan_rate_hz, title=f'Merged Point Waveform (point={74})',  ylabel='Phase (rad)',  ax=axes[1],)
plot_point_waveform(  extract_point_waveform(frame_data_phase, 60),    sample_rate_hz=scan_rate_hz, title=f'Merged Point Waveform (point={75})',  ylabel='Phase (rad)',  ax=axes[2],)
plot_point_waveform(  extract_point_waveform(frame_data_phase, 61),    sample_rate_hz=scan_rate_hz, title=f'Merged Point Waveform (point={76})',  ylabel='Phase (rad)',  ax=axes[3],)

(<Figure size 1440x960 with 4 Axes>,
 <Axes: title={'center': 'Merged Point Waveform (point=76)'}, xlabel='Time (s)', ylabel='Phase (rad)'>)

In [ ]:
from obspy import Stream, Trace
st= Stream()
tr70 = Trace(extract_point_waveform(frame_data_phase, 70))
# tr71 = Trace(extract_point_waveform(frame_data_phase, 71))
# tr72 = Trace(extract_point_waveform(frame_data_phase, 72))
# tr73 = Trace(extract_point_waveform(frame_data_phase, 73))
# tr74 = Trace(extract_point_waveform(frame_data_phase, 74))
# tr75 = Trace(extract_point_waveform(frame_data_phase, 75))
# tr76 = Trace(extract_point_waveform(frame_data_phase, 76))
# tr77 = Trace(extract_point_waveform(frame_data_phase, 77))
# tr70.stats.sampling_rate = scan_rate_hz
# tr71.stats.sampling_rate = scan_rate_hz
# tr72.stats.sampling_rate = scan_rate_hz
tr73.stats.sampling_rate = scan_rate_hz
tr74.stats.sampling_rate = scan_rate_hz
tr75.stats.sampling_rate = scan_rate_hz
tr76.stats.sampling_rate = scan_rate_hz
tr77.stats.sampling_rate = scan_rate_hz
st += tr70
st += tr71
st += tr72
st += tr73
st += tr74
st += tr75
st += tr76
st += tr77
# st.write('merged_traces.mseed', format='MSEED')
st.plot()

In [56]:
import numpy as np
from scipy.signal import butter, sosfiltfilt

def bandpass_filter_columns(X, fs, f_low, f_high, order=4, axis=0, padlen=None):
    """
    对矩阵按列做带通滤波（默认 axis=0，即每一列是一道信号，行是时间采样点）
    
    Parameters
    ----------
    X : np.ndarray, shape (N, M)
        输入数据，N=采样点数(480000)，M=道数(130)
    fs : float
        采样率(Hz)
    f_low, f_high : float
        带通下/上截止频率(Hz)，0 < f_low < f_high < fs/2
    order : int
        滤波器阶数（Butterworth）
    axis : int
        沿哪个轴滤波。axis=0: 每列一条信号；axis=1: 每行一条信号
    padlen : int or None
        sosfiltfilt 的 padlen。None 表示用默认值；若数据很长一般不需要改
    
    Returns
    -------
    Y : np.ndarray
        滤波后的数据，shape 同 X
    """
    if X.ndim != 2:
        raise ValueError("X must be 2D, expected shape (N, M).")

    nyq = 0.5 * fs
    if not (0 < f_low < f_high < nyq):
        raise ValueError(f"Require 0 < f_low < f_high < fs/2. Got nyq={nyq}.")

    # 用 SOS 数值更稳定
    sos = butter(order, [f_low/nyq, f_high/nyq], btype='bandpass', output='sos')

    # 零相位滤波；sosfiltfilt 支持 axis
    if padlen is None:
        Y = sosfiltfilt(sos, X, axis=axis)
    else:
        Y = sosfiltfilt(sos, X, axis=axis, padlen=padlen)
    return Y

# ===================== 用法示例 =====================
# X 形状 (480000, 130)
fs = 2000.0  # 举例：2000 Hz 采样率
f_low, f_high = 100, 999.0  # 举例：5-50 Hz 带通
Y = bandpass_filter_columns(frame_data_phase, fs, f_low, f_high, order=4, axis=0)




In [47]:
import numpy as np
import matplotlib.pyplot as plt

# X: shape (480000, 130)
# X = ...

fig, ax = plt.subplots(figsize=(8, 6), dpi=120)

im = ax.imshow(
    frame_data_phase.T,
    cmap="jet",        # 可改：'seismic', 'jet', 'gray' 等
    aspect="auto",         # 让 480000x130 能铺满显示
    origin="lower",        # 原点在左下（按需要改成 'upper'）
    interpolation="nearest",
    
    vmax=0.4,
    vmin=-0.4
)

ax.set_xlabel("Channel (0..129)")
ax.set_ylabel("Sample (0..479999)")
ax.set_title("Heatmap of Y (480000 x 130)")

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Amplitude")

plt.tight_layout()
plt.show()

In [27]:
print(frame_data_phase)

print(Y)

[[ 1.89835916e-02 -1.52923377e-01 -8.36045043e-02 ... -6.69420883e+00
  -1.71207068e+01 -3.96296856e+00]
 [-4.02682246e-03 -1.15914961e-01 -2.59825925e-02 ... -4.17332209e+00
  -1.43683737e+01 -1.77765036e+00]
 [ 8.10158328e-02 -5.92518162e-02 -7.67013801e-03 ... -2.31772395e+00
  -1.50588778e+01 -1.21475811e+00]
 ...
 [ 2.41513471e-01 -1.35953196e-01  1.10545864e-01 ... -6.40840031e+00
  -9.18518202e+00  1.55320295e-01]
 [ 2.52922801e-01 -1.07381932e-02 -5.36909661e-03 ... -5.27973950e+00
  -6.53716275e+00 -1.62654864e+00]
 [ 2.30775277e-01 -8.83024639e-02  9.94241640e-02 ... -6.24732741e+00
  -4.26248745e+00 -1.33527515e+00]]
[[-2.17783482e-03  9.70709914e-04  5.95121578e-04 ... -3.34947050e-02
   3.56604221e-02  2.04331343e-02]
 [-2.26002492e-02  2.94045142e-02  4.17992721e-02 ...  1.55921572e+00
   2.42685888e+00  1.02748821e+00]
 [ 5.77532337e-02  7.74821204e-02  4.67107374e-02 ...  2.36752787e+00
   1.44196058e+00  4.56519275e-01]
 ...
 [ 8.77993270e-02 -6.69785856e-02  6.2473133

In [11]:
# Cell 7: plot space-time view of the merged data
plot_space_time(
    frame_data_phase,
    frame_slice=space_time_frame_slice,
    point_slice=space_time_point_slice,
    sample_rate_hz=scan_rate_hz,
    cmap='jet',
    title='Merged Space-Time Plot',
    colorbar_label='Phase (rad)',
    vmin=-0.2,
    vmax=0.2,
)
plt.show()

## 第二次标定


In [7]:
# Cell 1: configure folder, matching rule, and display parameters
folder_path = Path(r"F:\2")
file_pattern = "000*.bin"
points_per_frame = None
scan_rate_hz = None
point_index = 100

space_time_frame_slice = slice(None)
space_time_point_slice = slice(None)
space_time_vmin = -0.2
space_time_vmax = 0.2

print('folder_path:', folder_path)
print('file_pattern:', file_pattern)
print('point_index:', point_index)

folder_path: F:\2
file_pattern: 000*.bin
point_index: 100


In [8]:
# Cell 2: read and concatenate raw int32 data from all matched files
frame_data_raw, matched_files = read_phase_bin_folder_raw(
    folder_path,
    pattern=file_pattern,
    points_per_frame=points_per_frame,
)

if points_per_frame is None:
    points_per_frame = infer_points_per_frame_from_filename(matched_files[0])
if scan_rate_hz is None:
    scan_rate_hz = infer_scan_rate_hz_from_filename(matched_files[0])

print('matched file count:', len(matched_files))
for path in matched_files:
    print('  ', path.name)
print('points_per_frame:', points_per_frame)
print('scan_rate_hz:', scan_rate_hz)
print('raw shape:', frame_data_raw.shape)
print('raw dtype:', frame_data_raw.dtype)
print('raw min/max:', frame_data_raw.min(), frame_data_raw.max())

# Cell 3: read the same folder directly as phase in radians
frame_data_phase, matched_files_phase = read_phase_bin_folder(
    folder_path,
    pattern=file_pattern,
    points_per_frame=points_per_frame,
)

print('same matched files:', matched_files_phase == matched_files)
print('phase shape:', frame_data_phase.shape)
print('phase dtype:', frame_data_phase.dtype)
print('phase min/max (rad):', frame_data_phase.min(), frame_data_phase.max())

matched file count: 855
   0000001-eDAS-2000Hz-0130pt-20260322T152836.652.bin
   0000173-eDAS-2000Hz-0130pt-20260322T153128.668.bin
   0000174-eDAS-2000Hz-0130pt-20260322T153129.661.bin
   0000175-eDAS-2000Hz-0130pt-20260322T153130.661.bin
   0000176-eDAS-2000Hz-0130pt-20260322T153131.661.bin
   0000177-eDAS-2000Hz-0130pt-20260322T153132.666.bin
   0000178-eDAS-2000Hz-0130pt-20260322T153133.660.bin
   0000179-eDAS-2000Hz-0130pt-20260322T153134.660.bin
   0000180-eDAS-2000Hz-0130pt-20260322T153135.672.bin
   0000181-eDAS-2000Hz-0130pt-20260322T153136.676.bin
   0000182-eDAS-2000Hz-0130pt-20260322T153137.659.bin
   0000183-eDAS-2000Hz-0130pt-20260322T153138.674.bin
   0000184-eDAS-2000Hz-0130pt-20260322T153139.666.bin
   0000185-eDAS-2000Hz-0130pt-20260322T153140.665.bin
   0000186-eDAS-2000Hz-0130pt-20260322T153141.663.bin
   0000187-eDAS-2000Hz-0130pt-20260322T153142.662.bin
   0000188-eDAS-2000Hz-0130pt-20260322T153143.659.bin
   0000189-eDAS-2000Hz-0130pt-20260322T153144.660.bin
   0

In [11]:
# Cell 7: plot space-time view of the merged data
plot_space_time(
    frame_data_phase,
    frame_slice=space_time_frame_slice,
    point_slice=space_time_point_slice,
    sample_rate_hz=scan_rate_hz,
    cmap='jet',
    title='Merged Space-Time Plot',
    colorbar_label='Phase (rad)',
    vmin=-0.4,
    vmax=0.4,
)
plt.show()

## 第三次标定

In [12]:
# Cell 1: configure folder, matching rule, and display parameters
folder_path = Path(r"F:\3")
file_pattern = "000*.bin"
points_per_frame = None
scan_rate_hz = None
point_index = 100

space_time_frame_slice = slice(None)
space_time_point_slice = slice(None)
space_time_vmin = -0.2
space_time_vmax = 0.2

print('folder_path:', folder_path)
print('file_pattern:', file_pattern)
print('point_index:', point_index)

folder_path: F:\3
file_pattern: 000*.bin
point_index: 100


In [13]:
# Cell 2: read and concatenate raw int32 data from all matched files
frame_data_raw, matched_files = read_phase_bin_folder_raw(
    folder_path,
    pattern=file_pattern,
    points_per_frame=points_per_frame,
)

if points_per_frame is None:
    points_per_frame = infer_points_per_frame_from_filename(matched_files[0])
if scan_rate_hz is None:
    scan_rate_hz = infer_scan_rate_hz_from_filename(matched_files[0])

print('matched file count:', len(matched_files))
for path in matched_files:
    print('  ', path.name)
print('points_per_frame:', points_per_frame)
print('scan_rate_hz:', scan_rate_hz)
print('raw shape:', frame_data_raw.shape)
print('raw dtype:', frame_data_raw.dtype)
print('raw min/max:', frame_data_raw.min(), frame_data_raw.max())

# Cell 3: read the same folder directly as phase in radians
frame_data_phase, matched_files_phase = read_phase_bin_folder(
    folder_path,
    pattern=file_pattern,
    points_per_frame=points_per_frame,
)

print('same matched files:', matched_files_phase == matched_files)
print('phase shape:', frame_data_phase.shape)
print('phase dtype:', frame_data_phase.dtype)
print('phase min/max (rad):', frame_data_phase.min(), frame_data_phase.max())

matched file count: 283
   0001603-eDAS-2000Hz-0130pt-20260322T155518.691.bin
   0001604-eDAS-2000Hz-0130pt-20260322T155519.690.bin
   0001605-eDAS-2000Hz-0130pt-20260322T155520.691.bin
   0001606-eDAS-2000Hz-0130pt-20260322T155521.692.bin
   0001607-eDAS-2000Hz-0130pt-20260322T155522.683.bin
   0001608-eDAS-2000Hz-0130pt-20260322T155523.682.bin
   0001609-eDAS-2000Hz-0130pt-20260322T155524.684.bin
   0001610-eDAS-2000Hz-0130pt-20260322T155525.694.bin
   0001611-eDAS-2000Hz-0130pt-20260322T155526.692.bin
   0001612-eDAS-2000Hz-0130pt-20260322T155527.690.bin
   0001613-eDAS-2000Hz-0130pt-20260322T155528.683.bin
   0001614-eDAS-2000Hz-0130pt-20260322T155529.683.bin
   0001615-eDAS-2000Hz-0130pt-20260322T155530.676.bin
   0001616-eDAS-2000Hz-0130pt-20260322T155531.677.bin
   0001617-eDAS-2000Hz-0130pt-20260322T155532.686.bin
   0001618-eDAS-2000Hz-0130pt-20260322T155533.688.bin
   0001619-eDAS-2000Hz-0130pt-20260322T155534.680.bin
   0001620-eDAS-2000Hz-0130pt-20260322T155535.691.bin
   0

In [16]:
# Cell 7: plot space-time view of the merged data
plot_space_time(
    frame_data_phase,
    frame_slice=space_time_frame_slice,
    point_slice=space_time_point_slice,
    sample_rate_hz=scan_rate_hz,
    cmap='jet',
    title='Merged Space-Time Plot',
    colorbar_label='Phase (rad)',
    vmin=-10,
    vmax=10,
)
plt.show()